In [44]:
!pip install bitarray
!pip install mmh3
from bitarray import bitarray
import hashlib
from hashlib import sha3_256, sha256, blake2b
import math 
import mmh3
import string

In [ ]:
#asked ChatGPT how large my bitarray needs to be to hold the large dataset
#isn't this very large and will cause hash collisions?

size = 360000
bits = bitarray(size)
bits.setall(0)
#bits n should be larger than data set size

In [ ]:
#Asked ChatGPT to explain bitarray library, fixed number of bits, salting, and hashes
#later functions weren't working and asked ChatGPT if I was adding words to the bloom filter right. Modified this code

words = []

with open('words.txt', 'r') as file:
    for line in file:
        word = line.strip()
        words.append(word)
        

In [60]:
class BloomFilter(object):
    #uses murmur3 hash function
    def __init__(self, items_count, fp_prob):
        #items_count number of items expected to be sotred in bloom filter
        #fp_prob false positive probability in decimal
        self.fp_prob = fp_prob #false positive in decimal; optional if size is fixed
        self.size = 3600000 #set size of bitarray
        self.hash_count = self.get_hash_count(self.size, items_count) #number of hash function
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
    def add(self, item):
        #add an item in the filter
        digests = []
        for i in range(self.hash_count): #create digest for given item, i seed for mmh3.hash; with different seed digest created is different
            digest = mmh3.hash(item, i) % self.size
            digests.append(digest)
            self.bit_array[digest] = True #set the bit True in bit_array
    def check(self, item):
        #check for item in filteer
        for i in range(self.hash_count):
            digest = mmh3.hash(item, i) % self.size
            if self.bit_array[digest] == False:
                #if any of bit is False, not present; else possibility it exists
                return False
        return True
    def get_hash_count(self, m, n):
        #return hash function for formula
        #m integer size of array
        #n integer number of items expected to be stored
        k = (m/n) * math.log(2)
        return int(k)


# https://www.geeksforgeeks.org/python/bloom-filters-introduction-and-python-implementation/


In [99]:
#for each word in the list, apply all three hash functions and set the corresponding bits in the bitarray.
#All return in [0, size] where size is some integer specified elsewhere

def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

for word in words: 
    index1 = my_hash(word)
    index2 = my_hash2(word)
    index3 = my_hash3(word)
    bits[index1] = 1
    bits[index2] = 1
    bits[index3] = 1
    
#Asked ChatGPT is I answered all parts of the question and modified code (define size) 


In [ ]:
#b. Create a function that checks all possible single-character substitutions for a given word using the Bloom filter. 
# Return words flagged by the filter as potential matches. 

#I need the function to take in each word, replace a single character, compare to the rest of list and return if it is a match. Repeat for all letter combinations

#Asked ChatGPT how to make a function that replces single character in given word
def single_char_changes(word): 
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

#Tested that the function does single character substitution for word given
print(single_char_changes('cat'))
len(single_char_changes('cat'))


['aat', 'bat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'cbt', 'cct', 'cdt', 'cet', 'cft', 'cgt', 'cht', 'cit', 'cjt', 'ckt', 'clt', 'cmt', 'cnt', 'cot', 'cpt', 'cqt', 'crt', 'cst', 'ctt', 'cut', 'cvt', 'cwt', 'cxt', 'cyt', 'czt', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'can', 'cao', 'cap', 'caq', 'car', 'cas', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz']


75

In [ ]:
#BloomFilter(number of items, false positive)
#https://www.geeksforgeeks.org/python/bloom-filters-introduction-and-python-implementation/
bloomf = BloomFilter(3600000, 0.05)

#add words to BloomFilter
for word in words:
    bloomf.add(word)

#create the function to check if word against bloom filter
def test_in_filter(test):
    return test in words if bloomf.check(test) else False

#test with words known in filter and known not in filter
print(test_in_filter('abandoner'))
print(test_in_filter('carolinejk'))

True
False


In [ ]:
#create full function
#Asked ChatGPT 
def spell_check(word):
    def single_char_changes(word): 
        replaced_words = []
        for i in range(len(word)):
            for letter in string.ascii_lowercase:
                if word[i] != letter:
                    changed = word[:i] + letter + word[i+1:]
                    replaced_words.append(changed)
        return replaced_words
    candidates = single_char_changes(word)
    matches = []
    for candidate in candidates:
        if candidate in words:
            print(f"{candidate} is a Match!")

In [ ]:
#test function
spell_check('cat')

eat is a Match!
qat is a Match!
rat is a Match!
xat is a Match!
yat is a Match!
zat is a Match!
cwt is a Match!
cag is a Match!
caw is a Match!
cay is a Match!


In [ ]:
# Implement a function to test how well the Bloom filter suggests corrections. 
# A suggestion list is considered "good" if it contains no more than three suggestions and includes the correct word.

def good_function()
    